# Day 30: Build a "Self-Querying Retriever"

Welcome to Day 30! Today we tackle a fundamental limitation of pure vector search: it struggles with exact constraints. If a user asks, "What are the Q3 2023 revenue reports for the EMEA region?", a standard semantic search will just look for documents with similar *meaning*, potentially returning Q2 reports or reports for North America because the text is semantically close.

A **Self-Querying Retriever** solves this by using an LLM to analyze the natural language query and extract structured metadata filters (e.g., `quarter == "Q3 2023"` and `region == "EMEA"`), alongside a clean semantic query.

## The "Why" and "How"

**Why we need it:**
Combining dense vector search (semantic similarity) with hard metadata filters ensures both high relevance and exact constraints.

**How it works (Architecture):**
1. **Query Construction:** An LLM is given a schema of available metadata fields. It parses the user's natural language query into a structured object using function calling/structured outputs.
2. **Translation:** The structured object is translated into the specific filter syntax of the target vector database (in our case, Qdrant).
3. **Execution:** The vector database executes the combined query: vector search on the semantic part, constrained by the metadata filters.

## Core Theory (Just-in-Time)

**Why we need it:**
Pure vector search relies on semantic similarity. If you ask for '2023 AI papers', a semantic search might return a 2022 paper just because it uses similar vocabulary. A Self-Querying Retriever fixes this by using an LLM to extract hard metadata constraints (like `year=2023`) and a clean semantic query ('AI papers'). This guarantees exact matching on metadata while preserving semantic richness.

**AI Security Implications:**
- **Prompt Injection:** An attacker might try to inject instructions into the natural language query to bypass filters. Always validate the extracted metadata fields and values against a strict schema (e.g., using Pydantic).
- **PII Leakage:** Ensure that the natural language queries sent to the LLM for extraction do not contain sensitive PII. If they do, they must be redacted before being sent.
- **Fallback Mechanisms:** LLMs can fail to extract correctly (hallucinations, parsing errors). Always implement a fallback to a default state (e.g., pure semantic search with no filters) if the extraction fails or returns an invalid schema.

## 1. Schema Definition & LLM Extraction
First, we define the structure we want the LLM to output using Pydantic. Then, we use LangChain and ChatGroq's `.with_structured_output()` to force the LLM to return this exact schema based on the natural language query.

In [ ]:
import os
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

### BASIC IMPLEMENTATION ###
# Isolating the core concept with minimal boilerplate.

class BasicFieldCondition(BaseModel):
    """Represents a single filter condition on a metadata field."""
    field: str = Field(description="The metadata field to filter on. e.g., 'year', 'category'")
    operator: Literal["eq", "gt", "lt", "gte", "lte"] = Field(description="The comparison operator.")
    value: Union[str, int, float, bool] = Field(description="The value to compare against.")

class BasicStructuredQuery(BaseModel):
    """The complete parsed query."""
    semantic_query: str = Field(description="The semantic part of the query.")
    filters: Optional[List[BasicFieldCondition]] = Field(default=None, description="Filter conditions.")

def basic_extract_query(natural_language_query: str) -> BasicStructuredQuery:
    """Uses an LLM to extract a StructuredQuery from natural language."""
    llm = ChatGroq(model="llama3-8b-8192", temperature=0)
    structured_llm = llm.with_structured_output(BasicStructuredQuery)
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert query analyzer. Extract semantic search queries and metadata filters. Available fields: 'year' (integer) and 'category' (string)."),
        ("human", "{query}")
    ])
    
    chain = prompt | structured_llm
    return chain.invoke({"query": natural_language_query})

# Test the basic extraction if API key is present
if os.environ.get("GROQ_API_KEY") and os.environ.get("GROQ_API_KEY") != "dummy":
    try:
        result = basic_extract_query("Show me AI papers from 2023")
        print("Basic Extraction:\n", result)
    except Exception as e:
        print("Extraction failed:", e)


In [ ]:
### MEDIUM IMPLEMENTATION ###
# Emphasizing clean OOP, state management, and how objects interact.

class QueryExtractor:
    """Encapsulates the LLM and extraction logic."""
    def __init__(self, model_name: str = "llama3-8b-8192"):
        self.llm = ChatGroq(model=model_name, temperature=0)
        self.structured_llm = self.llm.with_structured_output(BasicStructuredQuery)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are an expert query analyzer. Extract semantic search queries and metadata filters. Available fields: 'year' (integer) and 'category' (string)."),
            ("human", "{query}")
        ])
        self.chain = self.prompt | self.structured_llm

    def extract(self, query: str) -> BasicStructuredQuery:
        """Executes the extraction pipeline."""
        return self.chain.invoke({"query": query})

# Test medium implementation
if os.environ.get("GROQ_API_KEY") and os.environ.get("GROQ_API_KEY") != "dummy":
    extractor = QueryExtractor()
    try:
        print("Medium Extraction:\n", extractor.extract("Papers about physics before 2000"))
    except Exception as e:
        print("Extraction failed:", e)


In [ ]:
import logging
import re
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field, ValidationError
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

### ADVANCED IMPLEMENTATION ###
# Production-grade implementation with strict type hinting, docstrings, error handling, 
# and AI Security (PII protection, fallbacks).

class AdvancedFieldCondition(BaseModel):
    """Advanced, production-grade condition with strict typing."""
    field: Literal["year", "category"] = Field(description="Allowed metadata fields.")
    operator: Literal["eq", "gt", "lt", "gte", "lte"] = Field(description="Allowed operators.")
    value: Union[int, str] = Field(description="Comparison value.")

class AdvancedStructuredQuery(BaseModel):
    """Production-grade query schema."""
    semantic_query: str = Field(description="Cleaned semantic query.")
    filters: Optional[List[AdvancedFieldCondition]] = Field(default=None, description="Extracted filters.")

class ProductionQueryExtractor:
    """
    Advanced implementation handling PII redaction and providing robust fallbacks.
    """
    def __init__(self, model_name: str = "llama3-8b-8192"):
        self.llm = ChatGroq(model=model_name, temperature=0.0)
        self.structured_llm = self.llm.with_structured_output(AdvancedStructuredQuery)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a secure query analyzer. Extract semantic queries and metadata filters. Available fields: 'year' (integer) and 'category' (string)."),
            ("human", "{query}")
        ])
        self.chain = self.prompt | self.structured_llm
        
    def _redact_pii(self, text: str) -> str:
        """Redacts common PII like emails and phone numbers before sending to LLM."""
        # Redact emails
        text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '[REDACTED_EMAIL]', text)
        # Redact phone numbers (simple pattern for demonstration)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[REDACTED_PHONE]', text)
        return text

    def extract(self, query: str) -> AdvancedStructuredQuery:
        """Extracts structured query safely with fallbacks and PII protection."""
        safe_query = self._redact_pii(query)
        
        try:
            result = self.chain.invoke({"query": safe_query})
            logger.info("Successfully extracted structured query.")
            return result
        except ValidationError as ve:
            logger.error(f"Schema validation failed: {ve}. Potential prompt injection or hallucination.")
            raise ve
        except Exception as e:
            logger.error(f"LLM extraction failed: {e}.")
            raise e

# Replace the global 'StructuredQuery' and 'FieldCondition' for the rest of the notebook to use
StructuredQuery = AdvancedStructuredQuery
FieldCondition = AdvancedFieldCondition

# To test the fallback and PII redaction locally even without a real API key:
try:
    prod_extractor = ProductionQueryExtractor()
    print("Testing Advanced Extractor with PII:")
    print(prod_extractor.extract("Show me AI papers from 2023. My email is user@example.com"))
except Exception as e:
    print("Initialization or extraction failed. Make sure GROQ_API_KEY is set.", e)
    prod_extractor = None
# This will use the fallback if API key is dummy/missing

# Wrapper to maintain compatibility with the rest of the notebook pipeline
def extract_query(query: str) -> StructuredQuery:
    if prod_extractor is None:
        raise ValueError("Extractor not initialized.")
    return prod_extractor.extract(query)

# Test extraction for the rest of the notebook
try:
    parsed_output = extract_query("Show me AI papers from 2023")
except Exception as e:
    print("Extraction failed, using fallback:", e)
    parsed_output = StructuredQuery(semantic_query="AI papers", filters=[FieldCondition(field="year", operator="eq", value=2023)])


## 2. Filter Translation
We need to translate our Pydantic `StructuredQuery` into Qdrant's specific `Filter` models.

In [ ]:
from qdrant_client.models import Filter, FieldCondition as QdrantFieldCondition, MatchValue, Range

def translate_to_qdrant_filter(structured_query: StructuredQuery) -> Optional[Filter]:
    """
    Translates a StructuredQuery into a Qdrant Filter object.
    
    Args:
        structured_query: The parsed query from the LLM.
        
    Returns:
        A Qdrant Filter object if conditions exist, otherwise None.
    """
    if not structured_query.filters:
        return None
        
    qdrant_conditions = []
    
    for condition in structured_query.filters:
        if condition.operator == "eq":
            # Exact match
            qdrant_conditions.append(
                QdrantFieldCondition(
                    key=condition.field,
                    match=MatchValue(value=condition.value)
                )
            )
        else:
            # Range matches
            range_kwargs = {condition.operator: condition.value}
            qdrant_conditions.append(
                QdrantFieldCondition(
                    key=condition.field,
                    range=Range(**range_kwargs)
                )
            )
            
    return Filter(must=qdrant_conditions)

# Test translation
qdrant_filter = translate_to_qdrant_filter(parsed_output)
print("Translated Qdrant Filter:")
print(qdrant_filter)

## 3. End-to-End Retrieval Pipeline
Let's put it all together. We use an in-memory client and deterministic mock embeddings so this notebook runs independently.

In [ ]:
import hashlib
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, VectorParams, Distance

def embed_text(text: str, vector_size: int = 384) -> List[float]:
    """Generates a pseudo-random vector for text (mock embedding)."""
    hash_obj = hashlib.sha256(text.encode())
    hash_bytes = hash_obj.digest()
    vector = [(b / 128.0) - 1.0 for b in hash_bytes]
    if len(vector) < vector_size:
        vector = (vector * (vector_size // len(vector) + 1))[:vector_size]
    return vector[:vector_size]

def setup_and_populate_qdrant() -> QdrantClient:
    """Sets up an in-memory Qdrant client and populates it with sample data."""
    client = QdrantClient(":memory:")
    collection_name = "documents"
    
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    
    documents = [
        {"text": "Attention Is All You Need", "year": 2017, "category": "AI"},
        {"text": "BERT: Pre-training of Deep Bidirectional Transformers", "year": 2018, "category": "AI"},
        {"text": "GPT-4 Technical Report", "year": 2023, "category": "AI"},
        {"text": "A Brief History of Time", "year": 1988, "category": "Physics"}
    ]
    
    points = []
    for i, doc in enumerate(documents):
        points.append(
            PointStruct(
                id=i + 1,
                vector=embed_text(doc["text"]),
                payload=doc
            )
        )
        
    client.upsert(collection_name=collection_name, points=points)
    print(f"Ingested {len(points)} documents.")
    return client

client = setup_and_populate_qdrant()

def self_query_retrieve(query_string: str, client: QdrantClient):
    """Executes the complete self-query retrieval pipeline."""
    print(f"\n--- Original User Query: '{query_string}' ---")
    
    # 1. Use LLM to extract query and filters
    try:
        parsed_query = extract_query(query_string)
    except Exception as e:
        print(f"Extraction failed: {e}\nFalling back to hardcoded example.")
        parsed_query = StructuredQuery(
            semantic_query="AI papers",
            filters=[FieldCondition(field="year", operator="eq", value=2023)]
        )
        
    # 2. Translate filter
    qdrant_filter = translate_to_qdrant_filter(parsed_query)
    
    # 3. Embed semantic query
    query_vector = embed_text(parsed_query.semantic_query)
    
    # 4. Search using Qdrant (query_points + .points)
    results = client.query_points(
        collection_name="documents",
        query=query_vector,
        query_filter=qdrant_filter,
        limit=3
    )
    
    print("Results:")
    for hit in results.points:
        print(f"- {hit.payload.get('text')} (Year: {hit.payload.get('year')}, Category: {hit.payload.get('category')})")
        
# Execute the pipeline
self_query_retrieve("Show me AI papers from 2023", client)

## Common Pitfalls in Production

1. **Hallucinated Fields:** The LLM might invent metadata fields that don't exist in your index (e.g., filtering on `author_name` when the schema uses `author`). **Solution:** Provide explicit valid field names in the Pydantic schema descriptions.
2. **Type Mismatches:** The LLM might output a string `"2023"` when Qdrant expects an integer `2023` for range filters. **Solution:** Rely heavily on Pydantic's strict typing to coerce or reject invalid inputs before querying.
3. **Missing Filters vs. Over-filtering:** The LLM might apply too many filters, returning 0 results. **Solution:** Implement a fallback mechanism: if the filtered query returns nothing, try relaxing or removing the filters and re-running the semantic search.
4. **API Usage:** For Qdrant integrations (v1.19.0+), always use `QdrantClient.query_points()` and access results via the `.points` attribute, as `.search()` is deprecated.5. **PII Leakage:** Sending unredacted natural language queries to an LLM for filter extraction can leak sensitive data. **Solution:** Implement a PII redaction layer (like regex or a lightweight NER model) before passing the query to the prompt.


## Practical Lab: Support the 'IN' Operator

**Your Task:**
Currently, our `FieldCondition` only supports `eq` and numerical ranges (`gt`, `lt`, etc.). But what if a user asks: *"Show me papers from either 2017 or 2018?"*

1. Update `LabFieldCondition` to support an `"in"` operator.
2. Update the `value` type hint to accept a List of types as well.
3. Implement `lab_translate_to_qdrant_filter` to handle the `"in"` operator using Qdrant's `MatchAny` condition.

In [ ]:
from qdrant_client.models import MatchAny

class LabFieldCondition(BaseModel):
    field: str
    operator: Literal["eq", "gt", "lt", "gte", "lte", "in"]
    value: Union[str, int, float, bool, List[str], List[int]]

class LabStructuredQuery(BaseModel):
    semantic_query: str
    filters: Optional[List[LabFieldCondition]] = None

def lab_translate_to_qdrant_filter(structured_query: LabStructuredQuery) -> Optional[Filter]:
    """Implement translation for 'in' operator here."""
    if not structured_query.filters:
        return None
        
    qdrant_conditions = []
    for condition in structured_query.filters:
        if condition.operator == "eq":
            qdrant_conditions.append(
                QdrantFieldCondition(key=condition.field, match=MatchValue(value=condition.value))
            )
        elif condition.operator == "in":
            # YOUR CODE HERE: Implement MatchAny logic
            if isinstance(condition.value, list):
                qdrant_conditions.append(
                    QdrantFieldCondition(key=condition.field, match=MatchAny(any=condition.value))
                )
        else:
            range_kwargs = {condition.operator: condition.value}
            qdrant_conditions.append(
                QdrantFieldCondition(key=condition.field, range=Range(**range_kwargs))
            )
            
    return Filter(must=qdrant_conditions)

# Test your implementation
lab_query = LabStructuredQuery(
    semantic_query="Attention papers",
    filters=[
        LabFieldCondition(field="year", operator="in", value=[2017, 2018])
    ]
)

lab_filter = lab_translate_to_qdrant_filter(lab_query)
print("Lab Filter translated:")
print(lab_filter)

## Reference Links

- [LangChain Self-Querying Retriever Documentation](https://python.langchain.com/docs/modules/data_connection/retrievers/self_query/)
- [Qdrant Querying and Filtering](https://qdrant.tech/documentation/concepts/filtering/)
- [Pydantic v2 Documentation](https://docs.pydantic.dev/latest/)